In [ ]:
# ============================================================
# BRANCH B — STRUCTURAL CONVERSION
# D7 — UK National Audit Office — Delivering STEM Skills
#      for the Economy
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch B: Structural Conversion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!pip -q install pymupdf pymupdf4llm

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import re

import fitz
import pandas as pd
import pymupdf4llm

DOCUMENT_ID = "D7"
DOCUMENT_NAME = "UK National Audit Office — Delivering STEM skills for the economy"

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

CONVERSION_METHOD = (
    "pymupdf4llm page-aware Markdown conversion "
    "with complete-document retention"
)

LLM_INPUT_REPRESENTATION = "Structural Markdown"

SOURCE_FORMAT = ".pdf"
EXPECTED_SOURCE_SHA256 = "00cd2555312b220d7b4289144261aaba888336fb21b32b9ff53e96427a5f7eba"
EXPECTED_PAGE_COUNT = 12

EXPECTED_RECORD_COUNT = 59

EXPECTED_CATEGORY_COUNTS = {
    "Key fact": 10,
    "Policy context": 3,
    "Policy finding": 7,
    "Education pipeline statistic": 24,
    "Government initiative": 9,
    "Recommendation": 6
}

EXPECTED_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

STRING_OR_NULL_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

NUMERIC_OR_NULL_FIELDS = ["Value"]

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Source Location"
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

OUTPUT_DIR = Path("outputs_D7_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONVERSION_INTEGRITY_PATH = OUTPUT_DIR / "D7_branch_B_conversion_integrity.json"
REPRESENTATION_METADATA_PATH = OUTPUT_DIR / "D7_branch_B_representation.json"
STRUCTURAL_MARKDOWN_PATH = OUTPUT_DIR / "D7_branch_B_structural_markdown.md"
PROMPT_PATH = OUTPUT_DIR / "D7_branch_B_prompt.txt"
EXPERIMENT_METADATA_PRE_PATH = OUTPUT_DIR / "D7_branch_B_experiment_metadata_pre.json"
RAW_RESPONSE_PATH = OUTPUT_DIR / "D7_branch_B_raw_response.txt"
PARSED_EXTRACTION_PATH = OUTPUT_DIR / "D7_branch_B_parsed_extraction.json"
TECHNICAL_DIAGNOSTICS_PATH = OUTPUT_DIR / "D7_branch_B_technical_diagnostics.json"
EXPERIMENT_METADATA_PATH = OUTPUT_DIR / "D7_branch_B_experiment_metadata.json"
EXPERIMENT_SUMMARY_PATH = OUTPUT_DIR / "D7_branch_B_experiment_summary.json"

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected source pages:", EXPECTED_PAGE_COUNT)
print("Fixed Stage 1 reference records:", EXPECTED_RECORD_COUNT)
print("Expected fields:", len(EXPECTED_FIELDS))


In [ ]:
# ============================================================
# 1. Upload and verify the exact original D7 PDF
# ============================================================

uploaded = files.upload()

pdf_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".pdf")
]

if len(pdf_paths) != 1:
    raise ValueError("Upload exactly one original D7 PDF.")

SOURCE_PATH = pdf_paths[0]

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

if SOURCE_PATH.suffix.lower() != SOURCE_FORMAT:
    raise ValueError("Unexpected D7 source format.")

if not SOURCE_HASH_MATCH:
    raise ValueError("Uploaded D7 PDF does not match the frozen Stage 1 source identity.")

pdf_document = fitz.open(SOURCE_PATH)
PAGE_COUNT = len(pdf_document)
PAGE_COUNT_VALID = PAGE_COUNT == EXPECTED_PAGE_COUNT

if not PAGE_COUNT_VALID:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} pages; observed {PAGE_COUNT}."
    )

page_rows = []
source_page_text = {}

for page_number, page in enumerate(pdf_document, start=1):
    text = page.get_text("text") or ""
    source_page_text[page_number] = text

    page_rows.append({
        "Page Number": page_number,
        "Character Count": len(text),
        "Word Count": len(text.split()),
        "Text Extractable": bool(text.strip())
    })

page_df = pd.DataFrame(page_rows)

TEXT_EXTRACTABLE = bool(page_df["Text Extractable"].all())

if not TEXT_EXTRACTABLE:
    raise ValueError(
        "D7 is expected to contain machine-readable text. "
        "OCR is not introduced in Branch B."
    )

FULL_SOURCE_TEXT = "\n".join(source_page_text.values())

GROUNDING_MARKERS = {
    "key_facts": "Key facts",
    "summary": "Summary",
    "government_intervention": "Government intervention",
    "key_findings": "Key findings",
    "education_pipeline":
        "The performance of the education pipeline in delivering STEM skills",
    "latest_initiatives":
        "The latest initiatives designed to enhance the development of STEM skills",
    "value_for_money_conclusion": "Conclusion on value for money",
    "recommendations": "Recommendations"
}

GROUNDING_MARKER_STATUS = {
    key: marker.casefold() in FULL_SOURCE_TEXT.casefold()
    for key, marker in GROUNDING_MARKERS.items()
}

DOCUMENT_GROUNDING_VALID = all(GROUNDING_MARKER_STATUS.values())

if not DOCUMENT_GROUNDING_VALID:
    raise ValueError("One or more expected D7 source components are missing.")

SOURCE_CHECK = {
    "document_id": DOCUMENT_ID,
    "source_sha256": SOURCE_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "source_page_count": PAGE_COUNT,
    "source_page_count_valid": PAGE_COUNT_VALID,
    "machine_readable_text_layer": TEXT_EXTRACTABLE,
    "grounding_marker_checks": GROUNDING_MARKER_STATUS,
    "all_expected_components_present": DOCUMENT_GROUNDING_VALID
}

print(json.dumps(SOURCE_CHECK, indent=2, ensure_ascii=False))
display(page_df)


## D7 structural-conversion rule

The complete PDF is converted with `pymupdf4llm` using page chunks. Each converted page is retained and wrapped with an explicit `## Source Page N` boundary.

This differs from the older D7 Branch B notebook, which supplied only pages 6–12. The revised design retains pages 1–12 because Branch A exposes the complete original document.

No alternate plain-text fallback is used. If the defined structural conversion fails, the notebook stops rather than silently changing the representation method.

In [ ]:
# ============================================================
# 2. Convert the COMPLETE 12-page PDF to structural Markdown
# ============================================================

try:
    page_chunks = pymupdf4llm.to_markdown(
        str(SOURCE_PATH),
        page_chunks=True,
        write_images=False,
        show_progress=True
    )
except Exception as exc:
    raise RuntimeError(
        "D7 Branch B structural conversion failed. "
        "No fallback representation is used. "
        f"Original error: {exc}"
    )

if not isinstance(page_chunks, list):
    raise TypeError("Expected page_chunks=True to return a list.")

if len(page_chunks) != EXPECTED_PAGE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} converted page chunks; "
        f"observed {len(page_chunks)}."
    )

page_markdown = {}

for page_number, chunk in enumerate(page_chunks, start=1):
    text = chunk.get("text", "") if isinstance(chunk, dict) else str(chunk)

    if not text.strip():
        raise ValueError(
            f"Converted Markdown for source page {page_number} is empty."
        )

    page_markdown[page_number] = text.rstrip()

markdown_parts = [
    "# D7 — UK National Audit Office STEM Report",
    "",
    "> Complete structural conversion of the original 12-page PDF.",
    "> No pages have been removed from the Branch B representation.",
    ""
]

for page_number in range(1, EXPECTED_PAGE_COUNT + 1):
    markdown_parts.extend([
        f"## Source Page {page_number}",
        "",
        page_markdown[page_number],
        ""
    ])

STRUCTURAL_MARKDOWN = "\n".join(markdown_parts).rstrip() + "\n"

STRUCTURAL_MARKDOWN_PATH.write_text(
    STRUCTURAL_MARKDOWN,
    encoding="utf-8"
)

STRUCTURAL_MARKDOWN_SHA256 = sha256_file(STRUCTURAL_MARKDOWN_PATH)

print("Saved:", STRUCTURAL_MARKDOWN_PATH)
print("Representation SHA-256:", STRUCTURAL_MARKDOWN_SHA256)
print("Converted pages:", len(page_markdown))
print("Characters:", len(STRUCTURAL_MARKDOWN))


In [ ]:
# ============================================================
# 3. Conversion-integrity verification
# ============================================================

markdown_lower = STRUCTURAL_MARKDOWN.casefold()

page_boundary_checks = {
    str(page_number):
        f"## Source Page {page_number}" in STRUCTURAL_MARKDOWN
    for page_number in range(1, EXPECTED_PAGE_COUNT + 1)
}

component_checks = {
    key: marker.casefold() in markdown_lower
    for key, marker in GROUNDING_MARKERS.items()
}

# Representative source observations spanning the fixed extraction scope.
# These are integrity markers only; they do not construct model records.
REPRESENTATIVE_MARKERS = [
    "990",
    "442,000",
    "700,000",
    "2.6%",
    "30.9%",
    "42%",
    "9.4%",
    "21.2%",
    "6.9%",
    "19.9%",
    "17.6%",
    "67",
    "2,500",
    "15,000",
    "428",
    "810",
    "330",
    "Configure the labour market intelligence",
    "Fully embed a more structured approach to STEM across government"
]

representative_marker_checks = {
    marker:
        marker.casefold() in markdown_lower
    for marker in REPRESENTATIVE_MARKERS
}

CONVERSION_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_sha256": SOURCE_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "source_page_count": PAGE_COUNT,
    "converted_page_count": len(page_markdown),
    "all_source_pages_retained": all(page_boundary_checks.values()),
    "page_boundary_checks": page_boundary_checks,
    "expected_component_checks": component_checks,
    "all_expected_components_preserved": all(component_checks.values()),
    "representative_content_checks": representative_marker_checks,
    "all_representative_content_preserved":
        all(representative_marker_checks.values()),
    "conversion_method":
        CONVERSION_METHOD,
    "conversion_fallback_used": False,
    "part_specific_source_filtering_applied": False,
    "page_removal_applied": False,
    "ocr_applied": False,
    "page_cropping_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "numeric_calculation_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "normalisation_applied": False,
    "conversion_integrity_passed": all([
        SOURCE_HASH_MATCH,
        PAGE_COUNT == EXPECTED_PAGE_COUNT,
        len(page_markdown) == EXPECTED_PAGE_COUNT,
        all(page_boundary_checks.values()),
        all(component_checks.values()),
        all(representative_marker_checks.values())
    ])
}

CONVERSION_INTEGRITY_PATH.write_text(
    json.dumps(
        CONVERSION_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    CONVERSION_INTEGRITY,
    ensure_ascii=False,
    indent=2
))

if not CONVERSION_INTEGRITY["conversion_integrity_passed"]:
    raise ValueError("D7 Branch B conversion integrity failed.")


In [ ]:
# ============================================================
# 4. Preserve Branch B representation metadata
# ============================================================

REPRESENTATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "branch_name":
        BRANCH_NAME,

    "conversion_method":
        CONVERSION_METHOD,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,
    "representation_type":
        "Complete PDF converted to page-aware structural Markdown",
    "source_file": SOURCE_PATH.name,
    "source_format": SOURCE_FORMAT,
    "source_sha256": SOURCE_SHA256,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "source_page_count": PAGE_COUNT,
    "converted_page_count": len(page_markdown),
    "structural_conversion_applied": True,
    "conversion_fallback_used": False,
    "complete_source_document_retained": True,
    "page_boundaries_made_explicit": True,
    "page_removal_applied": False,
    "scope_enforced_by_prompt_not_representation_filtering": True,
    "ocr_applied": False,
    "normalisation_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"]
}

REPRESENTATION_METADATA_PATH.write_text(
    json.dumps(
        REPRESENTATION_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    REPRESENTATION_METADATA,
    ensure_ascii=False,
    indent=2
))


## Frozen extraction task

The following prompt mirrors the final D7 Branch A extraction task. Only representation-dependent wording and the branch identifier change.

It deliberately does **not** disclose:
- the expected 59 records;
- expected category counts;
- Stage 1 reference values;
- Validation A results.

This prevents Branch B from receiving stronger completion guidance than Branch A.

In [ ]:
# ============================================================
# 5. Create controlled Branch B extraction prompt
# ============================================================

BRANCH_B_PROMPT = """You are an information extraction assistant.

Extract the policy and quantitative records represented within the
defined scope of the attached structurally converted Markdown
representation of the report:

“Delivering STEM (science, technology, engineering and mathematics)
skills for the economy”.

Treat the attached structurally converted Markdown document as the only
source of information.

The complete 12-page source representation is attached.

Include records from the following defined source regions:

1. Every primary Key Facts item represented in the “Key facts” section.
   Treat explanatory or comparative wording embedded within a Key Facts
   item as context for that item rather than as an additional standalone
   record.

2. The principal policy-context statements represented in Summary
   paragraphs 1, 3 and 4 concerning:
   - the definition of STEM;
   - the main STEM skills-development routes;
   - departmental responsibilities for STEM skills.

3. The principal policy findings represented in Summary paragraphs
   7 to 12 and the conclusion on value for money in paragraph 21.

4. Every explicitly stated quantitative observation in Summary
   paragraphs 13 to 17 that belongs to the defined education-pipeline
   extraction scope.

5. The explicitly represented government-initiative observations in
   Summary paragraphs 18 to 20 concerning:

   - T levels and their career routes;
   - national colleges focusing on STEM skills;
   - the qualification level targeted by institutes of technology;
   - the maths and physics teacher supply package;
   - the target for recruiting additional maths and physics teachers;
   - the target for improving the skills of non-specialist teachers;
   - returning teachers recruited by the return to teaching pilot;
   - the recruitment target for that pilot;
   - returning teachers who completed the training provided.

Do not create an additional observation from comparative wording that
only describes the relationship between two already represented
quantities, such as a recruited number relative to its stated target.

6. Each recommendation represented in recommendations 22(a) to 24(f).

Exclude:

- publication metadata;
- contents-page entries;
- copyright and publisher information;
- contact details;
- website and social-media information;
- document prices;
- paragraph numbers and page numbers as observations;
- values appearing only in cross-references;
- qualitative explanatory details that are not one of the requested
  policy-context records, policy findings or recommendations;
- values that are not explicit source observations;
- calculated, derived or inferred values;
- Markdown representation metadata and page-boundary labels as
  observations.

For every included record extract exactly these fields:

- Category
- Statement or Section
- Metric
- Topic
- Value
- Unit
- Qualifier
- Reporting Period
- Source Location

Category:

Use exactly one of:

- Key fact
- Policy context
- Policy finding
- Education pipeline statistic
- Government initiative
- Recommendation

Statement or Section:

- Preserve a concise source-grounded statement or section label that
  identifies where the observation belongs.
- Do not introduce external interpretation.

Metric:

- Provide a concise source-grounded name for the quantitative metric
  or policy statement represented by the record.
- For qualitative policy records, use a concise label that identifies
  the policy concept or recommendation.

Topic:

- Preserve the relevant source-grounded STEM topic, population,
  programme, institution or policy area.
- Do not merge separate observations merely because they concern a
  similar topic.

Value:

- Use a JSON number for explicitly represented quantitative values.
- Use null for qualitative policy records that do not contain a
  primary quantitative value.
- Preserve explicitly negative values as negative numbers.
- When the source explicitly describes a quantitative change as a
  fall, decrease, decline or reduction, encode the Value as a negative
  number even when the printed percentage does not contain a minus sign.
- When the source explicitly describes a quantitative change as a rise,
  increase or growth, preserve the Value as positive.
- Do not calculate, derive, convert or infer values.
- Do not rescale, calculate or convert proportions or percentages.
  The directional-sign rule above is the only permitted sign encoding.

Unit:

- Preserve the source-grounded measurement unit.
- Use null when no explicit quantitative unit applies.
- Do not place approximation or inequality wording in Unit.

Qualifier:

- Preserve explicit approximation, inequality or threshold wording
  directly associated with a quantitative value, such as:
  “around”, “almost”, “over”, “more than”, “just over”, “minimum”
  or equivalent wording represented in the source.
- Use null when no explicit qualifier is associated with the value.

Reporting Period:

- Preserve explicitly associated years, academic years, durations,
  comparison periods or relative periods.
- Use null when no explicit reporting period applies.

Source Location:

Use concise physical-PDF source locations grounded in the source-page
boundaries represented in the Markdown, for example:

- PDF page 6 — Key facts
- PDF page 7 — Summary paragraph 1
- PDF page 7 — Summary paragraph 3
- PDF page 7 — Summary paragraph 4
- PDF page 8 — Summary paragraph 7
- PDF page 9 — Summary paragraph 8
- PDF page 9 — Summary paragraph 9
- PDF page 9 — Summary paragraph 10
- PDF page 9 — Summary paragraph 11
- PDF page 9 — Summary paragraph 12
- PDF page 10 — Summary paragraph 13
- PDF page 10 — Summary paragraph 14
- PDF page 10 — Summary paragraph 15
- PDF page 10 — Summary paragraph 16
- PDF page 11 — Summary paragraph 17
- PDF page 11 — Summary paragraph 18
- PDF page 11 — Summary paragraph 19
- PDF page 11 — Summary paragraph 20
- PDF page 11 — Summary paragraph 21
- PDF page 12 — Recommendation 22(a)
- PDF page 12 — Recommendation 22(b)
- PDF page 12 — Recommendation 23(c)
- PDF page 12 — Recommendation 23(d)
- PDF page 12 — Recommendation 24(e)
- PDF page 12 — Recommendation 24(f)

Additional extraction rules:

- Use the explicit structural cues and textual content represented in
  the Markdown.
- Preserve the physical PDF page references exposed by the source-page
  boundaries.
- Preserve repeated observations when the same or similar statistic
  is explicitly represented in different source sections.
- Do not deduplicate distinct source observations.
- Do not use external knowledge.
- Do not follow hyperlinks.
- Do not silently correct values, units or wording.
- Do not infer missing observations.
- Verify that all content within the defined source scope has been
  processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{
  "document_id": "D7",
  "branch": "B",
  "records": [
    {
      "Category": null,
      "Statement or Section": null,
      "Metric": null,
      "Topic": null,
      "Value": null,
      "Unit": null,
      "Qualifier": null,
      "Reporting Period": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
""".strip()

PROMPT_PATH.write_text(
    BRANCH_B_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(PROMPT_PATH)

print("Prompt saved:", PROMPT_PATH.name)
print("Prompt SHA-256:", PROMPT_SHA256)
print(BRANCH_B_PROMPT)


In [ ]:
# ============================================================
# 6. Create pre-extraction experiment metadata
# ============================================================

EXPERIMENT_METADATA_PRE = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_format": SOURCE_FORMAT,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "source_page_count": PAGE_COUNT,
    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "conversion_method":
        CONVERSION_METHOD,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "direct_document_ingestion": False,
    "structural_conversion_applied": True,
    "conversion_fallback_used": False,
    "complete_source_document_retained": True,
    "page_removal_applied": False,
    "scope_enforced_by_prompt_not_representation_filtering": True,
    "ocr_applied": False,
    "normalisation_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "expected_extraction_scope": {
        "expected_record_count": EXPECTED_RECORD_COUNT,
        "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
        "expected_fields": EXPECTED_FIELDS
    },
    "reference_expectations_disclosed_to_model": False,
    "conversion_integrity_file": CONVERSION_INTEGRITY_PATH.name,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "prompt_file": PROMPT_PATH.name,
    "prompt_sha256": PROMPT_SHA256,
    "expected_output_format":
        "JSON object with document_id, branch and records",
    "execution_environment": "Independent ChatGPT conversation",
    "content_validation_performed":
        False,
}

EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    EXPERIMENT_METADATA_PRE,
    ensure_ascii=False,
    indent=2
))


In [ ]:
# ============================================================
# 7. Download files for the independent LLM run
# ============================================================

for path in [
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH
]:
    files.download(path)

print(
    "\nIndependent execution instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D7_branch_B_structural_markdown.md.\n"
    "3. Submit the exact D7_branch_B_prompt.txt content once.\n"
    "4. Do not upload the original PDF, Stage 1 reference values, "
    "Validation A outputs, or expected counts.\n"
    "5. Save the complete first response exactly as returned as TXT.\n"
    "6. Do not correct, reorder, repair or regenerate the response."
)


In [ ]:
# ============================================================
# 8. Upload and preserve the untouched Branch B response
# ============================================================

uploaded_output = files.upload()

txt_paths = [
    Path(name)
    for name in uploaded_output
    if name.lower().endswith(".txt")
]

if len(txt_paths) != 1:
    raise ValueError(
        "Upload exactly one TXT file containing the complete "
        "D7 Branch B model response."
    )

UPLOADED_RAW_RESPONSE_PATH = txt_paths[0]

RAW_RESPONSE_TEXT = UPLOADED_RAW_RESPONSE_PATH.read_text(
    encoding="utf-8"
)

if not RAW_RESPONSE_TEXT.strip():
    raise ValueError("The uploaded model response is empty.")

RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = sha256_file(RAW_RESPONSE_PATH)

print("Raw response preserved:", RAW_RESPONSE_PATH.name)
print("Raw response SHA-256:", RAW_RESPONSE_SHA256)


In [ ]:
# ============================================================
# 9. Parse without repairing the model response
# ============================================================

valid_json = False
json_parsing_error = None
parsed_response = None

try:
    parsed_response = json.loads(RAW_RESPONSE_TEXT)
    valid_json = True
except json.JSONDecodeError as error:
    json_parsing_error = str(error)

top_level_object_valid = (
    valid_json
    and isinstance(parsed_response, dict)
)

document_id_present = (
    top_level_object_valid
    and "document_id" in parsed_response
)

document_id_correct = (
    document_id_present
    and parsed_response.get("document_id") == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch" in parsed_response
)

branch_correct = (
    branch_present
    and parsed_response.get("branch") == BRANCH
)

records_present = (
    top_level_object_valid
    and "records" in parsed_response
)

records_is_list = (
    records_present
    and isinstance(parsed_response.get("records"), list)
)

records_evaluable = all([
    valid_json,
    top_level_object_valid,
    records_present,
    records_is_list
])

extracted_records = (
    parsed_response["records"]
    if records_evaluable
    else []
)

observed_record_count = (
    len(extracted_records)
    if records_evaluable
    else None
)

parsed_extraction_created = False
parsed_extraction_sha256 = None

print("Valid JSON:", valid_json)
print("JSON parsing error:", json_parsing_error)
print("Records evaluable:", records_evaluable)
print("Observed record count:", observed_record_count)


In [ ]:
# ============================================================
# 10. Record-schema and field-type checks
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []

for record_index, record in enumerate(extracted_records):
    if not isinstance(record, dict):
        record_structure_issues.append({
            "record_index": record_index,
            "issue": "Record is not a JSON object"
        })
        continue

    observed_fields = list(record.keys())

    missing_fields = [
        field
        for field in EXPECTED_FIELDS
        if field not in record
    ]

    extra_fields = [
        field
        for field in observed_fields
        if field not in EXPECTED_FIELDS
    ]

    field_order_correct = observed_fields == EXPECTED_FIELDS

    if missing_fields or extra_fields or not field_order_correct:
        record_structure_issues.append({
            "record_index": record_index,
            "missing_fields": missing_fields,
            "extra_fields": extra_fields,
            "field_order_correct": field_order_correct,
            "observed_fields": observed_fields
        })

    for field in STRING_OR_NULL_FIELDS:
        value = record.get(field)

        if value is not None and not isinstance(value, str):
            field_type_issues.append({
                "record_index": record_index,
                "field": field,
                "observed_type": type(value).__name__
            })

    value = record.get("Value")

    if (
        value is not None
        and (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
        )
    ):
        field_type_issues.append({
            "record_index": record_index,
            "field": "Value",
            "observed_type": type(value).__name__
        })

    for field in MANDATORY_CONTENT_FIELDS:
        value = record.get(field)

        if value is None or value == "":
            missing_mandatory_values.append({
                "record_index": record_index,
                "field": field
            })

record_schema_valid = (
    len(record_structure_issues) == 0
    if records_evaluable
    else None
)

field_types_valid = (
    len(field_type_issues) == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(missing_mandatory_values) == 0
    if records_evaluable
    else None
)

records_with_type_issues = (
    len({
        issue["record_index"]
        for issue in field_type_issues
    })
    if records_evaluable
    else None
)

print("Record schema valid:", record_schema_valid)
print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)


In [ ]:
# ============================================================
# 11. Content/scope diagnostics — separate from schema validity
# ============================================================

if records_evaluable:
    record_count_valid = (
        observed_record_count == EXPECTED_RECORD_COUNT
    )

    observed_category_counts = dict(
        Counter(
            record.get("Category")
            for record in extracted_records
            if isinstance(record, dict)
        )
    )

    categories_valid = set(
        observed_category_counts
    ).issubset(ALLOWED_CATEGORIES)

    category_counts_valid = (
        observed_category_counts == EXPECTED_CATEGORY_COUNTS
    )

    # Exact-complete-record duplicate diagnostic.
    duplicate_counter = Counter(
        tuple(
            record.get(field)
            for field in EXPECTED_FIELDS
        )
        for record in extracted_records
        if isinstance(record, dict)
    )

    duplicate_records = [
        list(key)
        for key, count in duplicate_counter.items()
        if count > 1
    ]

    duplicate_record_count = len(duplicate_records)

    negative_values = [
        record.get("Value")
        for record in extracted_records
        if (
            isinstance(record, dict)
            and isinstance(record.get("Value"), (int, float))
            and not isinstance(record.get("Value"), bool)
            and record.get("Value") < 0
        )
    ]

    null_value_count = sum(
        1
        for record in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Value") is None
        )
    )

    qualifier_count = sum(
        1
        for record in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Qualifier") is not None
        )
    )

else:
    record_count_valid = None
    observed_category_counts = None
    categories_valid = None
    category_counts_valid = None
    duplicate_records = None
    duplicate_record_count = None
    negative_values = None
    null_value_count = None
    qualifier_count = None

CONTENT_DIAGNOSTICS = {
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": observed_record_count,
    "record_count_matches_reference": record_count_valid,
    "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts": observed_category_counts,
    "categories_valid": categories_valid,
    "category_counts_match_reference": category_counts_valid,
    "mandatory_fields_complete": mandatory_fields_complete,
    "missing_mandatory_value_count":
        len(missing_mandatory_values) if records_evaluable else None,
    "duplicate_complete_record_count": duplicate_record_count,
    "negative_values": negative_values,
    "null_value_count": null_value_count,
    "qualifier_count": qualifier_count
}

print(json.dumps(
    CONTENT_DIAGNOSTICS,
    ensure_ascii=False,
    indent=2
))


In [ ]:
# ============================================================
# 12. Determine technical/schema validity
# ============================================================

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])

parsed_extraction_created = False
parsed_extraction_sha256 = None

if structurally_evaluable:
    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get("document_id"),
        "branch":
            parsed_response.get("branch"),
        "records":
            extracted_records
    }

    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    parsed_extraction_created = True
    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )


TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "valid_json":
        bool(valid_json),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_present":
        bool(document_id_present),

    "document_id_correct":
        bool(document_id_correct),

    "branch_present":
        bool(branch_present),

    "branch_correct":
        bool(branch_correct),

    "records_present":
        bool(records_present),

    "records_is_list":
        bool(records_is_list),

    "records_evaluable":
        bool(records_evaluable),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        record_structure_issues
        if records_evaluable else None,

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issues":
        field_type_issues
        if records_evaluable else None,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "scope_complete":
        bool(record_count_valid)
        if record_count_valid is not None
        else False,

    "structurally_evaluable":
        bool(structurally_evaluable)
}

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D7_branch_B_technical_diagnostics.json"
)

TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    TECHNICAL_DIAGNOSTICS,
    ensure_ascii=False,
    indent=2
))


In [ ]:
# ============================================================
# 13. Create final experiment metadata and summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,
    "raw_response_file": RAW_RESPONSE_PATH.name,
    "raw_response_sha256": RAW_RESPONSE_SHA256,
    "parsed_extraction_file":
        PARSED_EXTRACTION_PATH.name if parsed_extraction_created else None,
    "parsed_extraction_sha256": parsed_extraction_sha256,
    "json_valid": valid_json,
    "records_evaluable": records_evaluable,
    "observed_record_count": observed_record_count,
    "observed_category_counts": observed_category_counts,
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,
    "content_validation_performed":
        False,
    "structurally_evaluable": bool(structurally_evaluable),
    "notes": (
        "Branch B converts the complete original 12-page D7 PDF to "
        "structural Markdown and supplies the complete representation "
        "to the model. No pages are removed based on extraction scope. "
        "Stage 1 reference values and expected counts are not supplied "
        "to the model. Accuracy is evaluated separately in Validation B."
    )
}

EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

EXPERIMENT_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "conversion_method":
        CONVERSION_METHOD,
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "structural_conversion_applied": True,
    "complete_source_document_retained": True,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": observed_record_count,
    "record_count_matches": record_count_valid,
    "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts": observed_category_counts,
    "category_counts_match": category_counts_valid,
    "valid_json": valid_json,
    "records_evaluable": records_evaluable,
    "record_schema_valid": record_schema_valid,
    "field_types_valid": field_types_valid,
    "structurally_evaluable": bool(structurally_evaluable),
    "scope_complete": record_count_valid,
    "duplicate_complete_record_count": duplicate_record_count,
    "negative_values": negative_values,
    "qualifier_count": qualifier_count,
    "parsed_extraction_created": parsed_extraction_created,
    "content_validation_performed":
        False,

    "notes": (
        "Content-level validation is performed separately "
        "in Validation B — D7."
    )
}

EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    EXPERIMENT_SUMMARY,
    ensure_ascii=False,
    indent=2
))


In [ ]:
# ============================================================
# 14. Final artefact inventory and downloads
# ============================================================

GENERATED_OUTPUTS = [
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    GENERATED_OUTPUTS.append(PARSED_EXTRACTION_PATH)

print("Generated D7 Branch B files:")

for path in GENERATED_OUTPUTS:
    print("-", path.name, "| exists:", path.exists())

for path in GENERATED_OUTPUTS:
    files.download(path)

